In [1]:
from pathlib import Path
import os
import pandas as pd
# from helpers.utils import dice_score
from helpers import paths
from helpers.paths import (
    RESOURCES_DIR, TRAIN_ROOT, PROJECT_ROOT
)
from helpers.shell_interface import open_itksnap_workspace_cmd

In [2]:
subject_sessions = pd.read_csv(RESOURCES_DIR/"subject-sessions.csv", index_col="sub")
subjects_file = PROJECT_ROOT / "training/roi_train2/subjects.txt"
with open(subjects_file, 'r') as f:
    subjects = [line.strip() for line in f.readlines()]

In [6]:
from collections import defaultdict
dataroot = Path("/mnt/z/3Tpioneer_bids")
label_names = [
    "lesion.t3m20/prl_mask_def_prob_LR.nii.gz", 
    "lesion.t3m20/prl_mask_def_prob_CH.nii.gz",
    "lesion.t3m20/prl_mask_def_prob_SRS.nii.gz",
    "lesion.t3m20/prl_mask_def_prob_SRS_CH.nii.gz",
]

labels_to_check = defaultdict(list)
for subid in subjects:
    sesid = subject_sessions.loc[int(subid), "ses"]
    subject_root = dataroot / f"sub-ms{subid}" / f"ses-{sesid}"
    for n in label_names:
        p = subject_root / n
        if p.exists():
            labels_to_check[subid].append(p)

In [7]:
for k,v in labels_to_check.copy().items():
    if len(v) < 2:
        labels_to_check.pop(k)

In [8]:
labels_to_check.keys()

dict_keys(['1011', '1038', '1074', '1080', '1215', '1293', '1348', '1396', '2026', '2041'])

In [12]:
import subprocess
subid = "1215"
labels = labels_to_check[subid]
subject_root = labels[0].parent.parent
images = [subject_root / im for im in ["flair.nii.gz", "phase.nii.gz"]]
for lab in labels:
    print(lab.name)
cmd = open_itksnap_workspace_cmd(images, labels, rename_root=("/mnt/z", "Z:/"))
# subprocess.run(cmd, shell=True)
print(cmd)

prl_mask_def_prob_CH.nii.gz
prl_mask_def_prob_SRS.nii.gz
itksnap -g Z:/3Tpioneer_bids/sub-ms1215/ses-20180429/flair.nii.gz -o Z:/3Tpioneer_bids/sub-ms1215/ses-20180429/phase.nii.gz -s Z:/3Tpioneer_bids/sub-ms1215/ses-20180429/lesion.t3m20/prl_mask_def_prob_CH.nii.gz Z:/3Tpioneer_bids/sub-ms1215/ses-20180429/lesion.t3m20/prl_mask_def_prob_SRS.nii.gz


- 2026: SRS and CH (LR not subsegmented); 2 PRL [2, 1]
- 1038: CH and LR; 2 PRL
- 1080: SRS and LR; 2 PRL
- 1215: SRS and CH; 2 PRL; [2, 3]
- 1348: LR and SRS; 2 PRL [1, 6]
- 1396: LR, SRS, and CH; 1 PRL [1]
- 1011: LR is not subsegmented
- 1074: SRS is not subsegmented
- 1293: Doesn't have unique labels (LR, LR_SRS, LR_SRS_CH)
- 2041: CH is not subsegmented

In [1]:
labels_to_check = {"images": [], "labels": []}
labels_to_check['images'].append(1)
labels_to_check

{'images': [1], 'labels': []}